# A First Colab Run — Assignment

**Week 1 · Assignment 1** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 100 &nbsp;·&nbsp; **Due:** *End of Week 1*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week1/assignments/week01_pipeline_warmup.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Continues Lesson 8 ("A First End-to-End Pipeline") — same dataset, same
pipeline. Every other week's assignment has three graded parts (Extend,
Decide, Recommend); this one only has the first. Parts B and C start in
Weeks 2 and 3.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_churn
from gbus564_coursework.utils.assignment_checks import check_equal, check_type

## Before You Start

- **This week: Part A only — 100 pts.** Every point comes from the
  checkpoints below passing. There's no Part B or C yet — those start in
  Weeks 2 and 3, once the course has actually taught what a modeling
  decision and a stakeholder recommendation are.
- **Self-checks are feedback, not the grade** — but this week, since
  Correctness is the *only* criterion, the checkpoints effectively are the
  grade. Green output means you're on track.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb`
  and submit that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus) — but the point of this week is
  proving *you* can run and lightly modify this pipeline yourself, not that
  an AI tool can.

## 1. Business Problem

Lesson 8 already introduced the business question this dataset answers:
which customers are likely to cancel, and who should the retention team
call first? This assignment doesn't ask a new question — it confirms you
can run that pipeline yourself, in your own Colab copy, and extend it:
build two new features, then measure whether they actually help.

**Business question:** same as Lesson 8 — rank customers by predicted
cancellation risk.

**Why it matters:** if you can't run the pipeline yourself, nothing later
in the course is buildable. And a feature that *sounds* useful isn't the
same as one that *is* — the only way to know is to fit the model with and
without it and compare.

## 2. Data — Part A: Extend

*Same dataset as Lesson 8 — loaded below, nothing to edit.*

In [ ]:
df = load_churn()
df.head()

### Data Prep — feature engineering

Lesson 7 covered *why* engineered features matter; here you build some.
You'll add three engineered features to a **copy** of the data, `df_fe`,
and leave the original `df` untouched — so later you can fit one model on
the original columns and one on the engineered version, and compare them
fairly.

**Example (given):** Lesson 8's `high_risk_segment` — customers on a
month-to-month contract *and* fiber-optic internet *and* paying by
electronic check, the highest-churn cohort in the dataset. Read how it's
built; your two features follow the same pattern.

In [ ]:
df_fe = df.copy()  # a copy — df itself stays exactly as loaded

df_fe["high_risk_segment"] = (
    (df_fe["Contract"] == "Month-to-month")
    & (df_fe["InternetService"] == "Fiber optic")
    & (df_fe["PaymentMethod"] == "Electronic check")
).map({True: "Yes", False: "No"})  # Yes/No text, like the dataset's other flags

df_fe.groupby("high_risk_segment")["Churn"].mean().round(2)

**Your turn — two features, both added to `df_fe`:**

1. **`early_tenure`** — `"Yes"` if the customer has been with the company
   **6 months or fewer** (`tenure <= 6`), otherwise `"No"`. New customers
   haven't built a habit yet; the first months are the riskiest window.
2. **`addon_count`** — how many of the six internet add-ons the customer
   pays for: `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`,
   `TechSupport`, `StreamingTV`, `StreamingMovies`. Count how many of those
   six columns are `"Yes"` for each customer (a number from 0 to 6). Each
   add-on is one more thing they'd have to replace if they switched
   providers.

*Hint for `addon_count`:* `df_fe[list_of_columns] == "Yes"` gives a table
of True/False, and `.sum(axis=1)` adds across each row (True counts as 1).

In [ ]:
# TODO: early_tenure — "Yes" if tenure <= 6, otherwise "No". Follow the
# high_risk_segment pattern above: a True/False condition, then .map().
# YOUR CODE HERE

# TODO: addon_count — how many of the six add-on columns are "Yes".
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]
# YOUR CODE HERE

df_fe[["tenure", "early_tenure"] + addons + ["addon_count"]].head()

In [ ]:
check_equal(int((df_fe["early_tenure"] == "Yes").sum()), 1481, "early_tenure flags the 1,481 customers with tenure <= 6")
check_equal(int(df_fe["addon_count"].sum()), 14353, "addon_count adds up to the right total across all customers")
check_equal(int(df_fe["addon_count"].max()), 6, "addon_count tops out at 6 (all six add-ons)")
check_equal("addon_count" in df.columns or "early_tenure" in df.columns, False, "original df is untouched — features went into df_fe")

### Split and encoding (given)

One split, so both models see **exactly the same customers** — the only
difference between them will be the three extra columns. Then one
encoding step per column set, *defined* here and fitted inside each
model's pipeline in Section 3 (Lesson 8's rule: define in Data, fit in
Model, on training data only).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

engineered = ["high_risk_segment", "early_tenure", "addon_count"]

X_fe, y = df_fe.drop(columns="Churn"), df_fe["Churn"]
X_fe_train, X_fe_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
# Same customers, original columns only.
X_train, X_test = X_fe_train.drop(columns=engineered), X_fe_test.drop(columns=engineered)

def make_preprocess(X):
    categorical_cols = X.select_dtypes(exclude="number").columns
    return ColumnTransformer(
        [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
        remainder="passthrough",
    )

preprocess = make_preprocess(X_train)        # original columns
preprocess_fe = make_preprocess(X_fe_train)  # original + engineered columns

print(f"Original columns: {X_train.shape[1]}   With engineered features: {X_fe_train.shape[1]}")

## 3. Model — Part A: Extend

*Baseline is given. Lesson 8 fit a `RandomForestClassifier` with its
default settings (100 trees) — here, fit it with **300** trees instead,
twice: once on the original columns (`model`) and once on the engineered
version (`model_fe`). Everything else stays the same as Lesson 8.*

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

# TODO: a RandomForestClassifier with n_estimators=300 (Lesson 8 used the
# default, 100), random_state=RANDOM_SEED, inside a Pipeline with
# `preprocess`. Fit it on X_train/y_train.
# YOUR CODE HERE

# TODO: model_fe — the same Pipeline, but with `preprocess_fe`, fitted on
# X_fe_train/y_train.
# YOUR CODE HERE


In [ ]:
check_type(model, Pipeline, "model is a fitted Pipeline")
check_equal(model.named_steps["clf"].n_estimators, 300, "model used 300 trees, not the default 100")
check_type(model_fe, Pipeline, "model_fe is a fitted Pipeline")
check_equal(model_fe.named_steps["clf"].n_estimators, 300, "model_fe used 300 trees too")
check_equal(model_fe.n_features_in_, X_fe_train.shape[1], "model_fe was trained on the engineered columns")

## 4. Evaluate — Part A: Extend

*Score both models against the baseline, same as Lesson 8.*

In [ ]:
from sklearn.metrics import accuracy_score

baseline_acc = accuracy_score(y_test, baseline.predict(X_test))

# TODO: each model's accuracy on its own test set — same pattern as
# baseline_acc above (model on X_test, model_fe on X_fe_test).
# YOUR CODE HERE

print(f"Baseline accuracy:                 {baseline_acc:.1%}")
print(f"Model, original columns:           {model_acc:.1%}")
print(f"Model, with engineered features:   {model_fe_acc:.1%}")

In [ ]:
check_equal(model_acc > baseline_acc, True, "original model beats the baseline")
check_equal(model_fe_acc > baseline_acc, True, "engineered model beats the baseline")

**Don't be surprised if the two models land within a point of each other —
or if the engineered one is slightly *worse*.** That's a real result, not a
bug. A random forest already discovers combinations like "month-to-month
*and* fiber *and* electronic check" on its own by stacking splits, so
handing it those combinations ready-made often adds little. Engineered
features pay off most for simpler models that *can't* find combinations by
themselves — Week 3's logistic regression is the first one you'll meet.
The habit this section is about: **never assume a change helped — fit it
both ways and measure.**

## 5. Interpret — Part A: Extend

*What is the engineered model actually keying off of? Same
feature-importance view as Lesson 8, for `model_fe` — given below, along
with where your three new features rank. Your fill-in: read the single most
important feature off it.*

In [ ]:
feature_names = model_fe.named_steps["prep"].get_feature_names_out()
importances = pd.Series(model_fe.named_steps["clf"].feature_importances_, index=feature_names)
importances.sort_values().tail(10).plot.barh(title="Top features driving the prediction (model_fe)")
plt.tight_layout()

# Where the engineered features rank, out of every column the model saw
ranks = importances.rank(ascending=False).astype(int)
print(ranks[ranks.index.str.contains("|".join(engineered))].sort_values().to_string())

In [ ]:
# TODO: the single highest-importance feature name, read off `importances`.
# YOUR CODE HERE

print(f"Top feature: {top_feature}")

In [ ]:
check_type(top_feature, str, "top_feature is a feature name (string)")
check_equal(top_feature in importances.index, True, "top_feature is actually one of the model's features")

## 6. Recommend

**Recommendation (same as Lesson 8):** rank customers by predicted
cancellation risk and route the top decile to the retention team.

*Nothing to fill in here yet.* Picking a threshold under a real cost
tradeoff (Part B) and writing that recommendation up for a stakeholder from
your own numbers (Part C) are next — Part B starts in Week 2, Part C in
Week 3. This section is here so you see the full six-part shape every
notebook this term follows, even in a week where only the first part is
graded.

## Rubric

*This week: one criterion, 100 pts. The full four-criterion rubric
(Correctness / Decision Quality / Justification / Communication) phases in
as Part B (Week 2) and Part C (Week 3) come online.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 100 | All checkpoints pass — both engineered features are built correctly on a copy of the data, both 300-tree models (original and engineered) are fitted and beat the baseline, and the top feature is correctly identified. |